In [ ]:
#NAME - DEVANSH SINGH RAWAT
#BATCH - 03 AIML
#SAP-ID - 500124378
#ROLL NO - R2142230767

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque

In [2]:
# 1. Load the model 
model = YOLO('yolov8n.pt')

# 2. Initialize the memory for the motion trail

trail = deque(maxlen=30)

In [3]:
def draw_motion_trail(frame, trail_points):
    """Draws a fading line connecting the points in the trail."""
    for i in range(1, len(trail_points)):
       
        if trail_points[i - 1] is None or trail_points[i] is None:
            continue

        # Calculate line thickness 
        thickness = int(np.sqrt(30 / float(i + 1)) * 4.5)
        
        # Draw the line segment in a light blue color
        cv2.line(frame, trail_points[i - 1], trail_points[i], (255, 100, 0), thickness)

In [4]:
# Start Video Capture
cap = cv2.VideoCapture(0)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1) # Mirror the view

        # Run YOLO inference
        results = model(frame, conf=0.5, verbose=False) 
        
        largest_area = 0
        best_center = None

        # Process detections
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                
                # Draw standard bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Find the center of the largest detection
                area = (x2 - x1) * (y2 - y1)
                if area > largest_area:
                    largest_area = area
                    best_center = (int((x1 + x2) / 2), int((y1 + y2) / 2))

        # Update tracking and draw
        trail.appendleft(best_center) # best_center is None if nothing detected
        draw_motion_trail(frame, trail)

        cv2.imshow("YOLO Gesture Tracking", frame)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    # This ensures the camera turns off even if you force-stop the Jupyter cell
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

In [1]:
from ultralytics import YOLO

# Load the blank model
model = YOLO('yolov8n.pt') 

# Train it and save to D: drive
results = model.train(
    data=r"D:\Anaconda\Pattern lab datasets\ASL DATASET\Sign Language Detection\data.yaml", # Update to your folder name
    epochs=30,  
    imgsz=640,  
    batch=8,    
    device='cpu',
    project=r"D:\Anaconda\Projects\ASL",    
    name='gesture_model_run'       

print("Training Complete! Check your D: drive.")

New https://pypi.org/project/ultralytics/8.4.27 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.9.25 torch-2.7.1+cu118 CPU (Intel Core i5-10300H 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Anaconda\Pattern lab datasets\ASL DATASET\Sign Language Detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

In [11]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque

# 1. Load YOUR custom trained model 

model = YOLO(r"D:\Anaconda\Projects\ASL\gesture_model_run\weights\best.pt") 

# 2. Initialize the memory for the motion trail
trail = deque(maxlen=30)

# 3. Start Video Capture
cap = cv2.VideoCapture(0)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Failed to grab frame.")
            break

        # Flip the frame horizontally for a mirror effect
        frame = cv2.flip(frame, 1)

        # Run inference using the CPU to bypass the GPU crash
        results = model(frame, conf=0.6, verbose=False, device='cpu') 
        
        largest_area = 0
        best_center = None

        # Process the detections
        for result in results:
            for box in result.boxes:
                # Get coordinates, confidence, and the custom class name
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                class_name = model.names[cls_id] 

                # Draw the bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Draw the gesture name and confidence score
                label = f"{class_name} {conf:.2f}"
                cv2.rectangle(frame, (x1, y1 - 25), (x1 + len(label) * 12, y1), (0, 255, 0), -1)
                cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

                # Find the center of the largest gesture detection for the trail
                area = (x2 - x1) * (y2 - y1)
                if area > largest_area:
                    largest_area = area
                    best_center = (int((x1 + x2) / 2), int((y1 + y2) / 2))

        # Update the trail memory
        if best_center is not None:
            trail.appendleft(best_center)
        else:
            trail.appendleft(None)

        # Draw the dynamic motion trail
        for i in range(1, len(trail)):
            if trail[i - 1] is None or trail[i] is None:
                continue
            
            # Make older points in the trail thinner
            thickness = int(np.sqrt(30 / float(i + 1)) * 4.5)
            cv2.line(frame, trail[i - 1], trail[i], (255, 100, 0), thickness)

        # Show the video window
        cv2.imshow("Custom Gesture Tracking & Motion Trail", frame)

        # Press 'q' on your keyboard to stop
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    # Safely shut down the camera and close windows
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)